In [6]:
from pathlib import Path
import time
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from dataset import get_flattened_data, class_names

BASE_DIR = Path.cwd().parent
RESULTS_DIR = BASE_DIR / "results"

In [7]:


import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split, PredefinedSplit


print("=" * 55)
print("  FNN Classifier — Fashion-MNIST")
print("=" * 55)
# ADDED VISUALS YAY!
# Load data
print("\n[1/3] Loading data...", end=" ", flush=True)
t0 = time.time()
X_train, y_train, X_val, y_val, X_test, y_test = get_flattened_data()
print(f"done ({time.time() - t0:.1f}s)")
print(f"      Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")

# Use a subset for fast hyperparameter search
X_search, _, y_search, _ = train_test_split(
    X_train, y_train,
    train_size=10000,
    stratify=y_train,
    random_state=42
)

X_combined = np.vstack((X_search, X_val))
y_combined = np.concatenate((y_search, y_val))

test_fold = np.concatenate([
    np.full(len(X_search), -1),
    np.zeros(len(X_val))
])

ps = PredefinedSplit(test_fold)

print(f"      Using {len(X_search):,} samples for architecture search")

# Hyperparameter search
architectures = [
    (128,),
    (256,),
    (128, 64),
    (256, 128),
    (256, 128, 64),
]

print(f"\n[2/3] Hyperparameter search — {len(architectures)} architectures (GridSearchCV)")
print("-" * 55)
t1 = time.time()

# 1. Define the base model with the "proper chance" early-stopping rules
base_model = MLPClassifier(
    activation="relu",
    solver="adam",
    max_iter=150,
    early_stopping=True,
    n_iter_no_change=5,
    random_state=42,
)

# 2. Define the parameters to search
param_grid = {
    'hidden_layer_sizes': architectures
}



# 3. Wrap it in GridSearchCV
# cv=3 means 3-fold cross validation.
# n_jobs=-1 is the magic bullet that trains multiple models at once.
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=ps,
    n_jobs=-1,
    verbose=2   # Prints scikit-learn's built-in progress since it absorbs the custom loop
)

# 4. Run the search
grid_search.fit(X_combined, y_combined)

# 5. Extract the data so your groupmate's plotting code works perfectly without changes
validation_accuracies = grid_search.cv_results_['mean_test_score']
best_arch = grid_search.best_params_['hidden_layer_sizes']
best_accuracy = grid_search.best_score_

elapsed = time.time() - t1
print(f"\n  Search complete in {elapsed:.1f}s")
print(f"  Best architecture found: {best_arch}")



  FNN Classifier — Fashion-MNIST

[1/3] Loading data... done (0.5s)
      Train: 51,000  Val: 9,000  Test: 10,000
      Using 10,000 samples for architecture search

[2/3] Hyperparameter search — 5 architectures (GridSearchCV)
-------------------------------------------------------
Fitting 1 folds for each of 5 candidates, totalling 5 fits

  Search complete in 43.0s
  Best architecture found: (256, 128)


In [9]:

print("\n  Detailed Architecture Results:")
print("  " + "-" * 53)
print(f"  {'Architecture':<18} | {'Val Accuracy':<12} | {'Fit Time':<10}")
print("  " + "-" * 53)

results = grid_search.cv_results_
for i in range(len(results['params'])):
    arch = str(results['params'][i]['hidden_layer_sizes'])
    acc = results['mean_test_score'][i]
    fit_time = results['mean_fit_time'][i]

    # Add a marker for the winning architecture
    marker = "  ◄ BEST" if acc == best_accuracy else ""

    print(f"  {arch:<18} | {acc:.4f}       | {fit_time:.1f}s{marker}")
print("  " + "-" * 53)


  Detailed Architecture Results:
  -----------------------------------------------------
  Architecture       | Val Accuracy | Fit Time  
  -----------------------------------------------------
  (128,)             | 0.8630       | 7.0s
  (256,)             | 0.8721       | 21.0s
  (128, 64)          | 0.8580       | 5.3s
  (256, 128)         | 0.8728       | 19.7s  ◄ BEST
  (256, 128, 64)     | 0.8696       | 17.0s
  -----------------------------------------------------


In [13]:
# Architecture bar chart
arch_labels = [str(a) for a in architectures]
plt.figure(figsize=(8, 4))

# Create the bars
bars = plt.bar(arch_labels, validation_accuracies)

plt.xlabel("Hidden layer architecture")
plt.ylabel("Validation accuracy")
plt.title("FNN Validation Accuracy by Architecture")
plt.xticks(rotation=15, ha="right")

# --- THE SCALING FIX ---
# Find the min and max accuracy to dynamically scale the y-axis
min_acc = min(validation_accuracies)
max_acc = max(validation_accuracies)

plt.ylim(min_acc - 0.01, max_acc + 0.01)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.001, f"{yval:.4f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fnn_arch_vs_accuracy.png")
plt.close()

In [14]:

# Final model on full data
print(f"\n[3/3] Retraining best architecture {best_arch} on full data (100 iters)...")
print("      verbose=True can see loss dropping every 10 iters", flush=True)
t2 = time.time()

final_model = MLPClassifier(
    hidden_layer_sizes=best_arch,
    activation="relu",
    solver="adam",
    max_iter=100,
    random_state=42,
    verbose=True,
)
final_model.fit(X_train, y_train)
print(f"      Retraining done ({time.time() - t2:.1f}s)")

# Test evaluation
print("\n  Evaluating on test set...", end=" ", flush=True)
test_predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print("done")

cm = confusion_matrix(y_test, test_predictions)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
)
display.plot(xticks_rotation=45)
plt.title("FNN Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fnn_confusion_matrix.png")
plt.close()

plt.figure()
plt.plot(final_model.loss_curve_)
plt.xlabel("Iteration")
plt.ylabel("Training loss")
plt.title("FNN Training Loss Curve")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fnn_loss_curve.png")
plt.close()

# Summary
print("\n" + "=" * 55)
print("  RESULTS")
print("=" * 55)
print(f"  Best architecture : {best_arch}")
print(f"  Best val accuracy : {best_accuracy:.4f}")
print(f"  Test accuracy     : {test_accuracy:.4f}")
print("  Plots saved to results/")
print("=" * 55)



[3/3] Retraining best architecture (256, 128) on full data (100 iters)...
      verbose=True can see loss dropping every 10 iters
Iteration 1, loss = 0.56260082
Iteration 2, loss = 0.38922353
Iteration 3, loss = 0.35044048
Iteration 4, loss = 0.32386310
Iteration 5, loss = 0.30118131
Iteration 6, loss = 0.28311875
Iteration 7, loss = 0.27124301
Iteration 8, loss = 0.25768484
Iteration 9, loss = 0.24876056
Iteration 10, loss = 0.23795036
Iteration 11, loss = 0.22816329
Iteration 12, loss = 0.22302348
Iteration 13, loss = 0.20854461
Iteration 14, loss = 0.20501261
Iteration 15, loss = 0.19974905
Iteration 16, loss = 0.19673861
Iteration 17, loss = 0.18277678
Iteration 18, loss = 0.17903401
Iteration 19, loss = 0.17596943
Iteration 20, loss = 0.17059619
Iteration 21, loss = 0.16273554
Iteration 22, loss = 0.16031113
Iteration 23, loss = 0.15434549
Iteration 24, loss = 0.15024979
Iteration 25, loss = 0.14813465
Iteration 26, loss = 0.14264823
Iteration 27, loss = 0.13513426
Iteration 28, 

C:\Users\William\PycharmProjects\cs178\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(



  RESULTS
  Best architecture : (256, 128)
  Best val accuracy : 0.8728
  Test accuracy     : 0.8891
  Plots saved to results/


In [12]:
from sklearn.metrics import classification_report

# Generate the full classification report
report = classification_report(
    y_test,
    test_predictions,
    target_names=class_names,
    digits=4
)

print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top     0.8009    0.8890    0.8427      1000
     Trouser     0.9819    0.9770    0.9794      1000
    Pullover     0.7746    0.8420    0.8069      1000
       Dress     0.9197    0.8700    0.8941      1000
        Coat     0.7848    0.8350    0.8091      1000
      Sandal     0.9797    0.9630    0.9713      1000
       Shirt     0.7862    0.6250    0.6964      1000
     Sneaker     0.9222    0.9720    0.9464      1000
         Bag     0.9740    0.9730    0.9735      1000
  Ankle boot     0.9700    0.9380    0.9537      1000

    accuracy                         0.8884     10000
   macro avg     0.8894    0.8884    0.8874     10000
weighted avg     0.8894    0.8884    0.8874     10000

